In [3]:
import os

In [4]:
def header(name, p, n, mem, time, email):
    """
    PARAMETERS
    ----------
    name: string
        name of batch job
    p: string
        node - options of 'node', 'seseml', 'sesempi', 'sesebig'
    n: string
        Number of processors needed for the job
    mem: string
        Total RAM required: i.e. '64GB'
    time: string
        Time needed for the job: 'Days-HH:MM:SS'
    email: string
        Email where info about execution is sent
    """
    string = '#!/bin/bash\n\n'
    string += '#SBATCH --job-name={}\n'.format(name)
    string += '#SBATCH -p {}\n'.format(p)
    string += '#SBATCH -n {}\n'.format(n)
    string += '#SBATCH --mem {}\n'.format(mem)
    string += '#SBATCH --time={}\n'.format(time)
    string += '#SBATCH --mail-type=BEGIN\n'
    string += '#SBATCH --mail-type=FAIL\n'
    string += '#SBATCH --mail-type=END\n'
    string += '#SBATCH --mail-user={}\n\n'.format(email)
    return string

def runPython(condaEnv, script, args, output=None):
    string = 'source ~/miniconda3/bin/activate {}\n\n'.format(condaEnv)
    string += 'python {}'.format(script)
    for arg in args:
        string += ' {}'.format(arg)
    if(output != None):
        string += ' > {}'.format(output)
    string += '\n'
    return string

def writeBatch(string, outFile):
    with open(outFile, 'w') as text_file:
        text_file.write(string)

# Generate observations batch writer

In [5]:
stcu_path = '/data/keeling/a/johntl3/b/Synthetic/IO/thesis/StratoCumulus'
cu_path = '/data/keeling/a/johntl3/b/Synthetic/IO/thesis/TradeCumulus'
tr_path = '/data/keeling/a/johntl3/b/Synthetic/IO/thesis/Transition'

fields = [stcu_path, cu_path, tr_path]
lengths = ['/len005', '/len007', '/len009', '/len011', '/len015', '/len019', '/len149']
expr = '/ex00'

p = 'node'
n = '4'
mem = '64GB'
time = '10-00:00:00'
email = 'johntl3@illinois.edu'

condaEnv = 'at3d_pure'
script = '/data/keeling/a/johntl3/b/Synthetic/scripts/gen_synthetic_obs_stochastic.py'

for f in fields:
    if(stcu_path in f):
        cat = 'stcu'
    elif(cu_path in f):
        cat = 'cu'
    elif(tr_path in f):
        cat = 'tr'
    for l in lengths:
        base = f + l
        os.makedirs(base, exist_ok=True)
        filename = base + expr + '_gen_synthetic_obs.sh'
        name = 'shdom{}{}'.format(l[-3:], cat)
        hdr = header(name, p, n, mem, time, email)
        arg = [base + expr + '.yml']
        output = base + expr + '.gen_obs.txt'
        body = runPython(condaEnv, script, arg, output)
        writeBatch(hdr + body, filename)


# Emulate obs batch writer

In [5]:
stcu_path = '/data/keeling/a/johntl3/b/Synthetic/IO/thesis/StratoCumulus'
cu_path = '/data/keeling/a/johntl3/b/Synthetic/IO/thesis/TradeCumulus'
tr_path = '/data/keeling/a/johntl3/b/Synthetic/IO/thesis/Transition'

fields = [stcu_path, cu_path, tr_path]
lengths = ['/len005', '/len007', '/len009', '/len011', '/len015', '/len019']
expr = '/ex00'

lg_len = '/len149'

p = 'node'
n = '4'
mem = '64GB'
time = '10-00:00:00'
email = 'johntl3@illinois.edu'

condaEnv = 'at3d_pure'
script = '/data/keeling/a/johntl3/b/Synthetic/scripts/emulate_obs_stochastic.py'

for f in fields:
    if(stcu_path in f):
        cat = 'stcu'
    elif(cu_path in f):
        cat = 'cu'
    elif(tr_path in f):
        cat = 'tr'
    lg_file = f + lg_len + expr
    for l in lengths:
        base = f + l
        os.makedirs(base, exist_ok=True)
        filename = base + expr + 'em_emulate_obs.sh'
        name = 'emu{}{}'.format(l[-3:], cat)
        hdr = header(name, p, n, mem, time, email)
        sm_file = base + expr
        args = [sm_file, lg_file]
        body = runPython(condaEnv, script, args)
        writeBatch(hdr + body, filename)


# AT3D Batch Writer

In [5]:
stcu_path = '/data/keeling/a/johntl3/b/Synthetic/IO/thesis/StratoCumulus'
cu_path = '/data/keeling/a/johntl3/b/Synthetic/IO/thesis/TradeCumulus'
tr_path = '/data/keeling/a/johntl3/b/Synthetic/IO/thesis/Transition'

fields = [stcu_path, cu_path, tr_path]
lengths = ['/len005', '/len007', '/len009', '/len011', '/len015', '/len019']
exprs = ['/ex00', '/ex00em']

p = 'node'
n = '4'
mem = '64GB'
time = '10-00:00:00'
email = 'johntl3@illinois.edu'

condaEnv = 'at3d_pure'
script = '/data/keeling/a/johntl3/b/Synthetic/scripts/botany_at3d.py'

for f in fields:
    if(stcu_path in f):
        cat = 'stcu'
    elif(cu_path in f):
        cat = 'cu'
    elif(tr_path in f):
        cat = 'tr'
    for l in lengths:
        base = f + l
        os.makedirs(base, exist_ok=True)
        for expr in exprs:
            if('em' in expr):
                em = 'em'
            else:
                em = ''
            filename = base + expr + '_at3d.sh'
            name = 'at3d{}{}_{}'.format(l[-3:], cat, em)
            hdr = header(name, p, n, mem, time, email)
            arg = [base + expr + '.yml']
            output = base + expr + '.at3d.txt'
            body = runPython(condaEnv, script, arg, output)
            writeBatch(hdr + body, filename)
